In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a GPU program that computes the dot product of two vectors containing 32-bit floating point numbers.
    The dot product is the sum of the products of the corresponding elements of two vectors.
</p>
<p>
    Mathematically, the dot product of two vectors $A$ and $B$ of length $n$ is defined as:
    $$
    A \cdot B = \sum_{i=0}^{n-1} A_i \cdot B_i = A_0 \cdot B_0 + A_1 \cdot B_1 + \ldots + A_{n-1} \cdot B_{n-1}
    $$
</p>
<h2>Implementation Requirements</h2>
<ul>
    <li>Use only GPU native features (external libraries are not permitted)</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>The final result must be stored in the output variable</li>
</ul>
<h2>Example 1:</h2>
<pre>Input:  A = [1.0, 2.0, 3.0, 4.0]
               B = [5.0, 6.0, 7.0, 8.0]
       Output: result = 70.0  (1.0*5.0 + 2.0*6.0 + 3.0*7.0 + 4.0*8.0)</pre>
<h2>Example 2:</h2>
<pre>Input:  A = [0.5, 1.5, 2.5]
               B = [2.0, 3.0, 4.0]
       Output: result = 15.5  (0.5*2.0 + 1.5*3.0 + 2.5*4.0)</pre>
<h2>Constraints</h2>
<ul>
    <li><code>A</code> and <code>B</code> have identical lengths</li>
    <li>1 ≤ <code>N</code> ≤ 100,000,000</li>

  <li>Performance is measured with <code>N</code> = 5</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, B, result are device pointers
extern "C" void solve(const float* A, const float* B, float* result, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, B, result are tensors on the GPU
@cute.jit
def solve(A: cute.Tensor, B: cute.Tensor, result: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(A: jax.Array, B: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    result: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, B, result are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, result: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# a, b, result are tensors on the GPU
def solve(a: torch.Tensor, b: torch.Tensor, result: torch.Tensor, n: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(name="Dot Product", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free")

    def reference_impl(self, A: torch.Tensor, B: torch.Tensor, result: torch.Tensor, N: int):
        assert A.shape == (N,)
        assert B.shape == (N,)
        assert result.shape == (1,)
        result[0] = torch.dot(A, B)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_float), "in"),
            "B": (ctypes.POINTER(ctypes.c_float), "in"),
            "result": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        A = torch.tensor([1.0, 2.0, 3.0, 4.0], device="cuda", dtype=dtype)
        B = torch.tensor([5.0, 6.0, 7.0, 8.0], device="cuda", dtype=dtype)
        result = torch.empty(1, device="cuda", dtype=dtype)
        return {
            "A": A,
            "B": B,
            "result": result,
            "N": 4,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []
        # basic_small
        tests.append(
            {
                "A": torch.tensor([1.0, 2.0, 3.0, 4.0], device="cuda", dtype=dtype),
                "B": torch.tensor([5.0, 6.0, 7.0, 8.0], device="cuda", dtype=dtype),
                "result": torch.empty(1, device="cuda", dtype=dtype),
                "N": 4,
            }
        )
        # all_zeros
        tests.append(
            {
                "A": torch.tensor([0.0] * 16, device="cuda", dtype=dtype),
                "B": torch.tensor([0.0] * 16, device="cuda", dtype=dtype),
                "result": torch.empty(1, device="cuda", dtype=dtype),
                "N": 16,
            }
        )
        # negative_numbers
        tests.append(
            {
                "A": torch.tensor([-1.0, -2.0, -3.0, -4.0], device="cuda", dtype=dtype),
                "B": torch.tensor([-5.0, -6.0, -7.0, -8.0], device="cuda", dtype=dtype),
                "result": torch.empty(1, device="cuda", dtype=dtype),
                "N": 4,
            }
        )
        # mixed_positive_negative
        tests.append(
            {
                "A": torch.tensor([1.0, -2.0, 3.0, -4.0], device="cuda", dtype=dtype),
                "B": torch.tensor([-1.0, 2.0, -3.0, 4.0], device="cuda", dtype=dtype),
                "result": torch.empty(1, device="cuda", dtype=dtype),
                "N": 4,
            }
        )
        # orthogonal_vectors
        tests.append(
            {
                "A": torch.tensor([1.0, 0.0, 0.0], device="cuda", dtype=dtype),
                "B": torch.tensor([0.0, 1.0, 0.0], device="cuda", dtype=dtype),
                "result": torch.empty(1, device="cuda", dtype=dtype),
                "N": 3,
            }
        )
        # medium_sized_vector
        tests.append(
            {
                "A": torch.empty(1000, device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "B": torch.empty(1000, device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "result": torch.empty(1, device="cuda", dtype=dtype),
                "N": 1000,
            }
        )
        # large_vector
        tests.append(
            {
                "A": torch.empty(10000, device="cuda", dtype=dtype).uniform_(-0.1, 0.1),
                "B": torch.empty(10000, device="cuda", dtype=dtype).uniform_(-0.1, 0.1),
                "result": torch.empty(1, device="cuda", dtype=dtype),
                "N": 10000,
            }
        )
        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 5
        A = torch.empty(N, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        B = torch.empty(N, device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        result = torch.zeros(1, device="cuda", dtype=dtype)
        return {
            "A": A,
            "B": B,
            "result": result,
            "N": N,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
